In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.sparse import issparse

In [ ]:
# Squeid

In [ ]:
Liver_sp_fil = scv.read('/home/jaehyunchoi/MASLD_collaboration/Spatial_data/Filtering_merged_adata.h5ad')

In [ ]:
Liver_sp_pre = scv.read('/home/jaehyunchoi/MASLD_collaboration/Spatial_data/Preprocessing_E1_adata_v2.h5ad')

In [ ]:
Liver_sp_fov = scv.read('/home/jaehyunchoi/MASLD_collaboration/Spatial_data/Preprocessing_E2_adata_v2.h5ad')

# Pre analysis

In [ ]:
Liver_sp_fil

In [ ]:
print(Liver_sp_fil.raw)

In [ ]:
print(Liver_sp_fil.obs)

In [ ]:
Liver_sp_fil.obsm['spatial']

In [ ]:
Liver_sp_fil.obs['predicted.id']

In [ ]:
sc.pp.normalize_total(Liver_sp_fil, target_sum=1e4)
sc.pp.log1p(Liver_sp_fil)

In [ ]:
sc.pp.highly_variable_genes(Liver_sp_fil, flavor="seurat", n_top_genes=2000)
sc.pp.pca(Liver_sp_fil, n_comps=50)
sc.pp.neighbors(Liver_sp_fil, n_neighbors=10, n_pcs=30)
sc.tl.umap(Liver_sp_fil)
sc.tl.leiden(Liver_sp_fil, resolution=0.5)   # resolution은 상황에 맞게 조절

In [ ]:
sc.pl.pca(Liver_sp_fil, color='fov')

In [ ]:
import matplotlib.pyplot as plt

plt.plot(Liver_sp_fil.uns['pca']['variance_ratio'][:50], marker='o')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Explained Variance')
plt.grid(True)
plt.show()

In [ ]:
sc.pl.umap(Liver_sp_fil, color = "predicted.id")

In [ ]:
sc.pl.umap(Liver_sp_fil, color = "fov")

In [ ]:
Liver_sp_fil

In [ ]:
print(Liver_sp_fil.obs['cell_id'])

In [ ]:
print(Liver_sp_fil.obs['fov'].unique())

In [ ]:
print(np.unique(Liver_sp_fil.obs['fov']))

In [ ]:
Liver_sp_fil_fov1

In [ ]:
Liver_sp_fil_fov1 = Liver_sp_fil[Liver_sp_fil.obs['fov'] == '1', :].copy()
Liver_sp_fil_fov2 = Liver_sp_fil[Liver_sp_fil.obs['fov'] == '2', :].copy()
Liver_sp_fil_fov3 = Liver_sp_fil[Liver_sp_fil.obs['fov'] == '3', :].copy()
Liver_sp_fil_fov4 = Liver_sp_fil[Liver_sp_fil.obs['fov'] == '4', :].copy()
Liver_sp_fil_fov5 = Liver_sp_fil[Liver_sp_fil.obs['fov'] == '5', :].copy()
print(Liver_sp_fil_fov1.shape)

In [ ]:
print(Liver_sp_fil_fov1.obsm['spatial'].shape)
print(Liver_sp_fil_fov1.obsm['spatial'][:5])


In [ ]:
print(Liver_sp_fil_fov1.obsm['spatial_fov'])

In [ ]:
# 1. 좌표 복사
Liver_sp_fil_fov1.obsm['spatial'] = Liver_sp_fil_fov1.obsm['spatial_fov']
Liver_sp_fil_fov1.obsm['spatial'] = Liver_sp_fil_fov1.obsm['spatial'] / 1000
# 2. 값 확인
print(Liver_sp_fil_fov1.obsm['spatial'].shape)
print(Liver_sp_fil_fov1.obsm['spatial'][:5])

# 3. 범위 확인
print(np.min(Liver_sp_fil_fov1.obsm['spatial'], axis=0))
print(np.max(Liver_sp_fil_fov1.obsm['spatial'], axis=0))

# 4. 다시 플롯
sc.pl.spatial(
    Liver_sp_fil_fov1, 
    color="predicted.id", 
    spot_size=0.1,   # 크기 늘려보기
    cmap="tab20"
)

In [ ]:
# 1. 좌표 복사
Liver_sp_fil_fov2.obsm['spatial'] = Liver_sp_fil_fov2.obsm['spatial_fov']
Liver_sp_fil_fov2.obsm['spatial'] = Liver_sp_fil_fov2.obsm['spatial'] / 3000
# 2. 값 확인
print(Liver_sp_fil_fov2.obsm['spatial'].shape)
print(Liver_sp_fil_fov2.obsm['spatial'][:5])

# 3. 범위 확인
print(np.min(Liver_sp_fil_fov2.obsm['spatial'], axis=0))
print(np.max(Liver_sp_fil_fov2.obsm['spatial'], axis=0))

# 4. 다시 플롯
sc.pl.spatial(
    Liver_sp_fil_fov2, 
    color="predicted.id", 
    spot_size=0.025,   # 크기 늘려보기
    cmap="tab20"
)

In [ ]:
import numpy as np
print(np.min(Liver_sp_fil_fov2.obsm['spatial'], axis=0))
print(np.max(Liver_sp_fil_fov2.obsm['spatial'], axis=0))

In [ ]:
df = Liver_sp_fil_fov2.obs.copy()
df["x"] = Liver_sp_fil_fov2.obsm["spatial"][:, 0]
df["y"] = Liver_sp_fil_fov2.obsm["spatial"][:, 1]
sns.scatterplot(data=df, x="x", y="y", hue="fov", s=1)
plt.gca().invert_yaxis()
plt.axis("equal")
plt.show()

In [ ]:
# 4. 다시 플롯
sc.pl.spatial(
    Liver_sp_fil, 
    color="predicted.id", 
    spot_size=0.1,   # 크기 늘려보기
    cmap="tab20"
)

# multiomics analysis processing 01 

In [ ]:
adata = scv.read('/home/jaehyunchoi/MASLD_collaboration/Spatial_data/Filtering_merged_adata.h5ad')
adata

In [ ]:
# spatial
df_spatial = pd.DataFrame(
    adata.X.toarray() if issparse(adata.X) else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

In [ ]:
df_spatial

In [ ]:
df_spatial['source'] = 'spatial'

In [ ]:
df_spatial.to_csv("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/df_spatial.csv", index=True)

# Squidpy analysis

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import squidpy as sq

In [ ]:
adata = sc.read('/home/jaehyunchoi/MASLD_collaboration/Spatial_data/Preprocessing_E2_adata_v2.h5ad')
adata

In [ ]:
sns.jointplot(
    data=adata.obs,
    x="total_counts",
    y="n_genes_by_counts",
    kind="scatter",
    alpha=0.2
)

In [ ]:
adata.obs

In [ ]:
adata.uns["spatial"]

In [ ]:
adata.obs[adata.obs["fov"] == "40"]["disease"].value_counts()

In [ ]:
import pandas as pd
pd.crosstab(adata.obs["fov"], adata.obs["disease"])

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["COL4A2"],
    library_key="fov",
    library_id=["40"],
    #seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["COL6A2"],
    library_key="fov",
    library_id=["40"],
    #seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["TNFRSF11B"], #OPG
    library_key="fov",
    library_id=["40"],
    #seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["SLC7A5"], #OPG
    library_key="fov",
    library_id=["40"],
    #seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["SLC7A5"], #OPG
    library_key="fov",
    library_id=["31"],
    #seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["GLUL"],
    library_key="fov",
    library_id=["40"],
    seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color="fov",
    library_id=[str(i) for i in range(1, 31)],  # '1'~'10'만 선택
    library_key="fov",  # <== 이거 꼭 필요함!
    size=10
)

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color="fov",
    library_id=[str(i) for i in range(31, 36)],  # '1'~'10'만 선택
    library_key="fov",  # <== 이거 꼭 필요함!
    size=10
)

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color="fov",
    library_id=[str(i) for i in range(36, 41)],  # '1'~'10'만 선택
    library_key="fov",  # <== 이거 꼭 필요함!
    size=10
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    library_key="fov",
    library_id=["36"],
    seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["GLUL","AXIN2","FABP4"],
    library_key="fov",
    library_id=["36"],
    seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["GLUL","AXIN2","FABP4"],
    library_key="fov",
    library_id=["37"],
    seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["GLUL","AXIN2"],
    library_key="fov",
    library_id=["36"],
    seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["GLUL","AXIN2"],
    library_key="fov",
    library_id=["37"],
    seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["GLUL","AXIN2"],
    library_key="fov",
    library_id=["38"],
    seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
adata.obs['predicted.id']

In [ ]:
adata.obs['predicted.id'].value_counts()

In [ ]:
adata.obs[adata.obs["fov"] == "36"]["predicted.id"].value_counts()

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["predicted.id"],
    groups = ["Endo4","T cells","Hepatocyte"],
    library_key="fov",
    library_id=["39"],
    #seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["predicted.id"],
    #groups = ["Endo3","Mes2","Hepatocyte"],
    library_key="fov",
    library_id=["39"],
    seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["predicted.id"],
    groups = ["Endo4","Mes2","Hepatocyte"],
    library_key="fov",
    library_id=["40"],
    #seg_contourpx=10,
    seg_cell_id="cell_ID",
    img=True,
    alpha = 0.3
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["predicted.id"],
    groups = ["Hepatocyte"],
    library_key="fov",
    library_id=["40"],
    #seg_contourpx=10,
    seg_cell_id="cell_ID",
    img=True,
    alpha = 0.3
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["predicted.id"],
    groups = ["Endo4","Endo2","Endo3"],
    library_key="fov",
    library_id=["36"],
    seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True,
    alpha = 0.3
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["GLUL"],
    library_key="fov",
    library_id=["36","38","40"],
    seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["LGR5"],
    library_key="fov",
    library_id=["36","38","40"],
    seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["AXIN2"],
    library_key="fov",
    library_id=["36","38","40"],
    seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)

In [ ]:
sq.pl.spatial_segment(
    adata,
    color=["GLUL"],
    library_key="fov",
    library_id=["38","39","40"],
    seg_contourpx=20,
    seg_cell_id="cell_ID",
    img=True
)